**CAP182-SS3 - Part B - STADIOEquities account-activation model**  
**Student:** Bernette Mogwera Lesejane - PGDip Data Science, STADIO (SOIT)

# 01 - Dataset loading

## Purpose
This notebook loads and reviews the 12 client tables before building the activation model.

- Check which tables are relevant, partly used, or not used
- Load only the useful tables
- Review the target definition and class balance
- Audit data quality issues before preprocessing

## Business problem
The main question is:

> Will this account make its first deposit within 90 days of registration?

Target: `activated` in `table01_registration_onboarding.csv`  
`activated = 1` when a first deposit occurs within 90 days; otherwise `0`.

In [1]:
%pip install -q xgboost imbalanced-learn seaborn

try:
    dbutils.library.restartPython()
except NameError:
    print("Not in Databricks — restart the kernel manually after installs complete.")

Note: you may need to restart the kernel to use updated packages.
Not in Databricks — restart the kernel manually after installs complete.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Setup, imports and shared settimgs

import os
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy import stats
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, auc, brier_score_loss,
                            classification_report, confusion_matrix, f1_score,
                            precision_recall_curve, precision_score, recall_score,
                            roc_auc_score, roc_curve)
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load and audit all 12 tables
# Check relevance and remove leakage
# Review target balance and data quality

# Configure project paths and shared settings
BASE = Path(os.environ.get("SS3_BASE", "/Workspace/Repos/mantshetlha@gmail.com/CAP182_Bernette"))


warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

DATA_DIR = Path(os.environ.get("SS3_DATA_DIR", "/Volumes/workspace/default/stadioequitiesdataextract"))  # client extract (12 CSV tables)
ARTIFACTS = BASE / "artifacts"    # cleaned data, split assignments, fitted preprocessor, models
RESULTS = BASE / "results"        # metrics, tables and figures
FIGS = RESULTS / "figures"
for _p in (ARTIFACTS, RESULTS, FIGS):
    _p.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = "activated"
ID = "account_id"


def read_table(name):
    """Read a client table exactly as supplied.

    keep_default_na=False stops pandas turning the literal category "None"
    (e.g. investor_experience_level = None, 1,250 rows) into a missing value.
    Only genuinely empty cells become NaN.
    """
    return pd.read_csv(DATA_DIR / f"{name}.csv", keep_default_na=False, na_values=[""])

print("Setup complete. Data folder:", DATA_DIR.resolve())

# Load and inventory all client tables
TABLE_NAMES = [p.stem for p in sorted(DATA_DIR.glob("table*.csv"))]
raw = {name: read_table(name) for name in TABLE_NAMES}

inventory = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "distinct_accounts": df[ID].nunique() if ID in df else np.nan,
        "rows_per_account (max)": df[ID].value_counts().max() if ID in df else np.nan,
        "exact_duplicate_rows": int(df.duplicated().sum()),
    }
    for name, df in raw.items()
])
inventory.to_csv(RESULTS / "01_table_inventory.csv", index=False)
print(f"Tables found: {len(raw)}")
inventory

# Record table relevance and leakage decisions
RELEVANCE = pd.DataFrame([
    ("table01_registration_onboarding", "USE", "Target + onboarding funnel; drop first_deposit_date, days_to_first_deposit (leakage)."),
    ("table02_initial_session_behaviour", "USE", "First-session and early engagement (pre-activation behaviour)."),
    ("table03_demographics_identity", "USE", "Stated goal, experience and region. gender excluded as a protected attribute."),
    ("table04_communication_support", "USE", "Onboarding / promotional e-mail engagement and support-contact counts."),
    ("table05_pre_deposit_interaction", "USE (partial)", "Drop pre_deposit_window_days, total_pre_deposit_minutes (censored by deposit); account_balance_before_first_action is constant 0."),
    ("table06_inapp_messaging_response", "USE", "Push and in-app engagement: a lever the client can pull."),
    ("table07_support_tickets", "NOT USED", "No timestamp (cannot prove pre-deposit); counts duplicate Table 04 exactly."),
    ("table08_marketing_campaigns", "USE (partial)", "Aggregated to account level; drop conversion_within_30days, campaign_roi_percent (outcome-derived)."),
    ("table09_compliance_kyc", "USE", "KYC/AML status; snapshot at extract date, tested in sensitivity analysis."),
    ("table10_feedback_satisfaction", "NOT USED", "barrier_to_activation exists only for non-activators (leakage); ~31% response, non-random."),
    ("table11_device_technical", "USE", "Sign-up device, OS, browser, connection, VPN / ad-blocker flags."),
    ("table12_post_deposit_activity", "NOT USED", "Exists only for activated accounts; post-deposit behaviour (leakage)."),
], columns=["table", "verdict", "reason"])
RELEVANCE.to_csv(RESULTS / "01_table_relevance.csv", index=False)

# Select the tables for analysis
used_tables = RELEVANCE.loc[RELEVANCE.verdict.str.startswith("USE"), "table"].tolist()
excluded_tables = RELEVANCE.loc[RELEVANCE.verdict.str.startswith("NOT"), "table"].tolist()
print("Tables used     :", len(used_tables))
print("Tables not used :", excluded_tables)
RELEVANCE

used = {name: raw[name] for name in used_tables}

population = set(used["table01_registration_onboarding"][ID])
for name, df in used.items():
    ids = set(df[ID])
    print(f"{name:<36} rows={len(df):>5}  accounts={len(ids):>5}  "
          f"in-target-population={len(ids & population):>5}")

t1_raw = used["table01_registration_onboarding"]
t1 = t1_raw.drop_duplicates()
assert t1[ID].is_unique, "duplicate account_id remains after exact-duplicate removal"

# Measure unique accounts and target balance
class_counts = t1[TARGET].value_counts().sort_index()
summary = pd.DataFrame({
    "class": ["Not activated (0)", "Activated (1)"],
    "accounts": [class_counts.get(0, 0), class_counts.get(1, 0)],
})
summary["share"] = summary["accounts"] / summary["accounts"].sum()

print(f"Raw rows in table01        : {len(t1_raw):,}")
print(f"Exact duplicates removed   : {len(t1_raw) - len(t1):,}")
print(f"Modelling instances        : {len(t1):,} unique accounts")
print(f"Majority : minority ratio  : {class_counts.max() / class_counts.min():.2f} : 1")
print()
print(summary.to_string(index=False, formatters={"share": "{:.1%}".format}))

summary.to_csv(RESULTS / "01_class_balance.csv", index=False)

# Audit missing values
missing_rows = []
for name, df in used.items():
    for col in df.columns:
        n_missing = int(df[col].isna().sum())
        if n_missing:
            missing_rows.append({"table": name, "column": col, "missing": n_missing,
                                 "missing_share": round(n_missing / len(df), 3)})
missing_report = pd.DataFrame(missing_rows)
print("Columns with missing values (true blanks):")
print(missing_report.to_string(index=False))

# Find inconsistent category labels
def spelling_variants(series):
    s = series.dropna().astype(str)
    raw_distinct = s.nunique()
    collapsed = s.str.replace(r"\s+", " ", regex=True).str.strip().str.lower().nunique()
    return raw_distinct, collapsed

variant_rows = []
for name, df in used.items():
    for col in df.select_dtypes(include=["object", "string"]).columns:
        if col in (ID,) or col.endswith("_date") or col.endswith("_timestamp"):
            continue
        raw_n, clean_n = spelling_variants(df[col])
        if raw_n > clean_n:
            variant_rows.append({"table": name, "column": col,
                                 "distinct_raw": raw_n, "distinct_after_normalising": clean_n,
                                 "spurious_variants": raw_n - clean_n})
variant_report = pd.DataFrame(variant_rows).sort_values("spurious_variants", ascending=False)
print("\nCategory columns with spelling / casing / spacing variants:")
print(variant_report.to_string(index=False))

# Check registration date formats
patterns = {
    "ISO  yyyy-MM-dd HH:mm:ss": r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$",
    "slash yyyy/MM/dd":          r"^\d{4}/\d{2}/\d{2}$",
    "slash dd/MM/yyyy":          r"^\d{2}/\d{2}/\d{4}$",
    "slash MM/dd/yyyy HH:mm":    r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$",
}
reg = t1_raw["registration_date"].astype(str)
date_rows = [{"format": k, "rows": int(reg.str.match(p).sum())} for k, p in patterns.items()]
date_rows.append({"format": "unrecognised", "rows": int(len(reg) - sum(r["rows"] for r in date_rows))})
date_report = pd.DataFrame(date_rows)
print("\nRegistration date formats (table 01):")
print(date_report.to_string(index=False))

missing_report.to_csv(RESULTS / "01_missing_values.csv", index=False)
variant_report.to_csv(RESULTS / "01_category_variants.csv", index=False)
date_report.to_csv(RESULTS / "01_date_formats.csv", index=False)

# Save the dataset audit summary
dataset_summary = pd.DataFrame([
    ("Relevant tables loaded", len(used_tables)),
    ("Tables not used (relevance / leakage)", len(excluded_tables)),
    ("Raw rows, table 01", len(t1_raw)),
    ("Exact duplicate rows removed", len(t1_raw) - len(t1)),
    ("Modelling instances (unique accounts)", len(t1)),
    ("Class 0 · not activated within 90 days", int(class_counts.get(0, 0))),
    ("Class 1 · activated within 90 days", int(class_counts.get(1, 0))),
    ("Majority : minority ratio", round(class_counts.max() / class_counts.min(), 3)),
], columns=["item", "value"])
dataset_summary.to_csv(RESULTS / "01_dataset_summary.csv", index=False)
print("Saved results/01_dataset_summary.csv")
dataset_summary

Setup complete. Data folder: C:\Volumes\workspace\default\stadioequitiesdataextract
Tables found: 0
Tables used     : 9
Tables not used : ['table07_support_tickets', 'table10_feedback_satisfaction', 'table12_post_deposit_activity']


KeyError: 'table01_registration_onboarding'